In [3]:
import cv2
import numpy as np

cap = cv2.VideoCapture('robbery.mp4')

ret, frame1 = cap.read()
ret, frame2 = cap.read()

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def balance_light(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_eq = clahe.apply(l)
    lab_eq = cv2.merge((l_eq, a, b))
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

while cap.isOpened():
    if not ret:
        break

    small1 = cv2.resize(frame1, (0, 0), fx=0.5, fy=0.5)
    small2 = cv2.resize(frame2, (0, 0), fx=0.5, fy=0.5)

    frame1_bal = balance_light(small1)
    frame2_bal = balance_light(small2)

    diff = cv2.absdiff(frame1_bal, frame2_bal)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

    blur = cv2.bilateralFilter(gray, 15, 9, 9)

    threshold = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 11, 3
    )

    contour, _ = cv2.findContours(threshold, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contour:
        if cv2.contourArea(cnt) < 5000:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(frame1_bal, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(frame1_bal, 'Suspicious', (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.imshow('detection', frame1_bal)

    frame1 = frame2
    ret, frame2 = cap.read()

    if cv2.waitKey(3) == 27:
        break

cap.release()
cv2.destroyAllWindows()